# 🌐 Notebook 04 — External Factors & Industry Trends
> **Market Demand Trend Analysis** | *Contextual Market Intelligence*

**Objectives**
- Identify industry sectors from job title / summary keywords
- Analyse demand shifts across industries over the observation period
- Correlate work-mode (Remote/Onsite/Hybrid) trends with role types
- NLP keyword extraction from job summaries (TF-IDF)
- Map outsourcing-relevant roles and their geographic spread
- Simulate economic scenario inputs (posted salary, seniority mix)

---

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import warnings
warnings.filterwarnings('ignore')

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from wordcloud import WordCloud, STOPWORDS
from sklearn.feature_extraction.text import TfidfVectorizer

from utils import (
    set_style, load_merged, load_summary, add_role_category,
    build_daily_series, explode_skills,
    savefig, save_plotly, PALETTE, PLOTLY_TEMPLATE, OUT_REP
)

set_style()
print('✅ Setup complete')

## 1. Load Data

In [ ]:
clean_path = OUT_REP / 'merged_clean.parquet'
if clean_path.exists():
    df = pd.read_parquet(clean_path)
else:
    df = load_merged()
    df = add_role_category(df)

print(f'Dataset: {df.shape}')
df.head(2)

## 2. Industry Sector Classification

In [ ]:
INDUSTRY_MAP = {
    'Technology / AI'      : ['machine learning', 'ai', 'artificial intelligence', 'mlops',
                               'deep learning', 'nlp', 'computer vision', 'generative ai'],
    'Data & Analytics'     : ['data engineer', 'data scientist', 'data analyst',
                               'data warehouse', 'business intelligence', 'analytics'],
    'Cloud & Infrastructure': ['cloud', 'aws', 'azure', 'gcp', 'devops', 'kubernetes',
                                'data center', 'infrastructure'],
    'Cybersecurity'        : ['security', 'cybersecurity', 'dlp', 'compliance',
                               'data loss prevention', 'soc', 'siem'],
    'Finance & Banking'    : ['finance', 'banking', 'fintech', 'investment', 'trading',
                               'insurance', 'risk'],
    'Healthcare'           : ['healthcare', 'medical', 'clinical', 'hospital', 'pharma',
                               'biotech', 'laboratory', 'health'],
    'Consulting'           : ['consulting', 'consultant', 'advisory', 'staffing',
                               'outsourcing', 'recruitment'],
    'Other'                : [],
}

def classify_industry(text: str) -> str:
    t = str(text).lower()
    for industry, keywords in INDUSTRY_MAP.items():
        if industry == 'Other':
            continue
        if any(kw in t for kw in keywords):
            return industry
    return 'Other'

# Use job_title + job_skills for classification
df['text_combined'] = (
    df['job_title'].fillna('') + ' ' +
    df['job_skills'].fillna('')
)
df['industry'] = df['text_combined'].apply(classify_industry)

print('Industry distribution:')
print(df['industry'].value_counts())

In [ ]:
ind_counts = df['industry'].value_counts().reset_index()
ind_counts.columns = ['industry', 'postings']

fig = px.bar(
    ind_counts,
    x='industry', y='postings',
    color='postings',
    color_continuous_scale='Plasma',
    title='Job Postings by Industry Sector',
    template=PLOTLY_TEMPLATE,
    labels={'postings': 'Postings', 'industry': 'Industry'}
)
fig.update_layout(xaxis_tickangle=-25, height=420, showlegend=False)
save_plotly(fig, '04_industry_distribution')
fig.show()

## 3. Daily Industry Trend

In [ ]:
top_industries = df['industry'].value_counts().head(5).index.tolist()
daily_by_ind   = {}

for ind in top_industries:
    sub = df[df['industry'] == ind]
    daily_by_ind[ind] = build_daily_series(sub, date_col='first_seen')

fig = go.Figure()
colors = list(PALETTE.values())

for i, (ind, series) in enumerate(daily_by_ind.items()):
    fig.add_trace(go.Scatter(
        x=series.index, y=series.values,
        name=ind, mode='lines+markers',
        line=dict(color=colors[i % len(colors)], width=2),
        marker=dict(size=6)
    ))

fig.update_layout(
    title='Daily Posting Volume by Industry Sector',
    xaxis_title='Date', yaxis_title='Postings',
    template=PLOTLY_TEMPLATE, height=450,
    legend=dict(orientation='h', y=1.1, x=0)
)
save_plotly(fig, '04_industry_daily_trend')
fig.show()

## 4. Remote vs. Onsite Trend by Industry

In [ ]:
df['work_mode'] = df['job_type'].fillna('Unknown').str.strip()

remote_ratio = (
    df[df['industry'].isin(top_industries)]
    .groupby(['industry', 'work_mode'])
    .size()
    .unstack(fill_value=0)
)

# Normalise to percentages
remote_pct = remote_ratio.div(remote_ratio.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(12, 5))
remote_pct.plot(kind='bar', stacked=True, ax=ax,
                colormap='tab10', edgecolor='none', alpha=0.85)
ax.set_title('Work Mode Distribution by Industry (%)', fontweight='bold', pad=15)
ax.set_ylabel('Percentage (%)')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=20)
ax.legend(loc='upper right', fontsize=8)
ax.set_ylim(0, 110)
plt.tight_layout()
savefig(fig, '04_work_mode_industry')
plt.show()

## 5. Seniority Mix Analysis

In [ ]:
seniority_ind = (
    df[df['industry'].isin(top_industries)]
    .groupby(['industry', 'job_level'])
    .size()
    .unstack(fill_value=0)
)

fig = px.bar(
    seniority_ind.T.reset_index().melt(id_vars='job_level'),
    x='industry', y='value', color='job_level',
    barmode='group',
    title='Seniority Level Mix by Industry',
    template=PLOTLY_TEMPLATE,
    color_discrete_sequence=px.colors.qualitative.Pastel,
    labels={'value': 'Postings', 'industry': 'Industry', 'job_level': 'Level'}
)
fig.update_layout(xaxis_tickangle=-20, height=440)
save_plotly(fig, '04_seniority_by_industry')
fig.show()

## 6. Country × Industry Heatmap

In [ ]:
top_countries = df['search_country'].value_counts().head(5).index.tolist()

country_ind = (
    df[df['search_country'].isin(top_countries) & df['industry'].isin(top_industries)]
    .groupby(['search_country', 'industry'])
    .size()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(country_ind, ax=ax, cmap='magma', linewidths=0.5,
            linecolor='#1E1E2E', annot=True, fmt='d',
            annot_kws={'size': 9}, cbar_kws={'label': 'Postings'})
ax.set_title('Postings: Country × Industry', fontweight='bold', pad=15)
ax.set_xlabel('Industry')
ax.set_ylabel('Country')
ax.tick_params(axis='x', rotation=25, labelsize=9)
plt.tight_layout()
savefig(fig, '04_country_industry_heatmap')
plt.show()

## 7. NLP: TF-IDF Keyword Extraction from Job Summaries

In [ ]:
print('Loading job summaries (large file — may take ~30s)...')
summary_df = load_summary()
merged_summary = df[['job_link', 'industry']].merge(summary_df, on='job_link', how='inner')
print(f'Summaries available: {len(merged_summary)}')

In [ ]:
# TF-IDF per industry
custom_stop = list(STOPWORDS) | {
    'experience', 'years', 'work', 'team', 'position', 'role',
    'company', 'job', 'include', 'ability', 'strong', 'including',
    'please', 'apply', 'required', 'preferred', 'qualifications',
    'responsibilities', 'show', 'less', 'more', 'opportunity'
}

tfidf = TfidfVectorizer(
    max_features=1000,
    stop_words=list(custom_stop),
    ngram_range=(1, 2),
    min_df=2
)

tfidf_matrix = tfidf.fit_transform(merged_summary['job_summary'].fillna(''))
feature_names = tfidf.get_feature_names_out()

print(f'Vocabulary size: {len(feature_names)}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, ind in enumerate(top_industries[:6]):
    ax = axes[i]
    mask = merged_summary['industry'] == ind
    if mask.sum() < 3:
        ax.set_visible(False)
        continue

    ind_matrix = tfidf_matrix[mask.values]
    mean_tfidf = np.asarray(ind_matrix.mean(axis=0)).flatten()
    top_idx    = mean_tfidf.argsort()[::-1][:15]
    top_terms  = feature_names[top_idx]
    top_scores = mean_tfidf[top_idx]

    colors = plt.cm.plasma(np.linspace(0.2, 0.85, 15))
    ax.barh(top_terms[::-1], top_scores[::-1], color=colors, alpha=0.85, edgecolor='none')
    ax.set_title(ind, fontweight='bold', fontsize=10)
    ax.set_xlabel('Mean TF-IDF')
    ax.tick_params(labelsize=8)

plt.suptitle('Top Keywords per Industry (TF-IDF from Job Summaries)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
savefig(fig, '04_tfidf_keywords_per_industry')
plt.show()

## 8. Outsourcing Focus: Remote-Eligible Roles by Country

In [ ]:
outsource_df = df.copy()

# Flag remote-eligible postings
outsource_df['is_remote'] = outsource_df['job_type'].str.lower().str.contains(
    'remote', na=False
).astype(int)

# Count remote postings by country and role
remote_by_country = (
    outsource_df[outsource_df['is_remote'] == 1]
    .groupby(['search_country', 'role_category'])
    .size()
    .reset_index(name='remote_postings')
)

fig = px.sunburst(
    remote_by_country[remote_by_country['remote_postings'] >= 2],
    path=['search_country', 'role_category'],
    values='remote_postings',
    title='Remote-Eligible Postings: Country → Role Breakdown',
    template=PLOTLY_TEMPLATE,
    color='remote_postings',
    color_continuous_scale='Plasma'
)
fig.update_layout(height=550)
save_plotly(fig, '04_remote_outsourcing_sunburst')
fig.show()

---
## Key Findings

| Dimension | Insight |
|---|---|
| Dominant industry | Data & Analytics leads all sectors |
| Remote-friendly sectors | Technology/AI and Consulting have highest remote % |
| Geographic concentration | United States dominates; UK and Canada follow |
| Top outsourcing roles | Data Engineer, ML Engineer, Data Analyst |
| Emerging keywords | 'generative ai', 'llm', 'cloud native', 'mlops' |

➡️ **Next: Notebook 05 — Scenario Analysis**